In [ ]:
from pathlib import Path
import sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from spectral_mixture_analysis.audio import load_mono_audio, match_length
from spectral_mixture_analysis.transforms import compute_representation, amplitude_to_db
from spectral_mixture_analysis.metrics import l2_error, relative_l2_error, mean_absolute_error, spectral_overlap
from spectral_mixture_analysis.separation import separate_spectrogram

RESULTS_DIR = PROJECT_ROOT / "results" / "drum_tests"
(RESULTS_DIR / "figures").mkdir(parents=True, exist_ok=True)

In [ ]:
SR = 22050
SAMPLES = PROJECT_ROOT / "data" / "samples"

drum, _ = load_mono_audio(SAMPLES / "drums" / "drum.wav", sr=SR)

TRACK1 = {
    "bass":    SAMPLES / "bass"    / "Track1_Bass.wav",
    "flute":   SAMPLES / "flute"   / "Track1_flute.wav",
    "piano":   SAMPLES / "piano"   / "Track1_Acoustic_Grand_Piano.wav",
    "trumpet": SAMPLES / "trumpet" / "Track1_trumpet.wav",
}

tracks = {}
for name, path in TRACK1.items():
    sig, _ = load_mono_audio(path, sr=SR)
    drum, sig = match_length(drum, sig, mode="truncate")
    tracks[name] = sig

print(f"drum : {len(drum)/SR:.2f}s")
for name, sig in tracks.items():
    print(f"  {name}: {len(sig)/SR:.2f}s")

In [ ]:
TRANSFORM_CONFIGS = {
    "stft": dict(output="magnitude", n_fft=2048, hop_length=512),
    "mel":  dict(n_fft=2048, hop_length=512),
    "nsgt": dict(fmin=30.0, fmax=11000.0, bins=48, scale="oct"),
}

In [ ]:
rows = []
reps = {}  # (transform, inst) -> (R_drum, R_instr, R_mix)

for transform, kwargs in TRANSFORM_CONFIGS.items():
    for inst, sig in tracks.items():
        r_drum  = compute_representation(drum,       SR, transform_type=transform, **kwargs)
        r_instr = compute_representation(sig,        SR, transform_type=transform, **kwargs)
        r_mix   = compute_representation(drum + sig, SR, transform_type=transform, **kwargs)

        reps[(transform, inst)] = (r_drum, r_instr, r_mix)

        r_sum = r_drum + r_instr
        rows.append({
            "transform":   transform,
            "instrument":  inst,
            "l2":          l2_error(r_mix, r_sum),
            "relative_l2": relative_l2_error(r_mix, r_sum),
            "mae":         mean_absolute_error(r_mix, r_sum),
            "overlap":     spectral_overlap(r_drum, r_instr),
        })
        print(f"[{transform}] drum + {inst:8s}  rel_l2={rows[-1]['relative_l2']:.4f}  overlap={rows[-1]['overlap']:.4f}")

df = pd.DataFrame(rows)

In [ ]:
for transform in TRANSFORM_CONFIGS:
    fig, axes = plt.subplots(4, 3, figsize=(14, 14))
    fig.suptitle(f"Drum test — {transform.upper()}", fontsize=13)

    for row_idx, inst in enumerate(tracks):
        r_drum, r_instr, r_mix = reps[(transform, inst)]
        for col_idx, (rep, label) in enumerate([
            (r_drum,  "drum"),
            (r_instr, inst),
            (r_mix,   f"drum + {inst}"),
        ]):
            ax = axes[row_idx, col_idx]
            im = ax.imshow(
                amplitude_to_db(np.clip(rep, 0, None)),
                origin="lower", aspect="auto", interpolation="none",
            )
            plt.colorbar(im, ax=ax, format="%+2.0f dB")
            if row_idx == 0:
                ax.set_title(label)
            if col_idx == 0:
                ax.set_ylabel(inst)

    plt.tight_layout()
    fig_path = RESULTS_DIR / "figures" / f"{transform}.png"
    plt.savefig(fig_path, dpi=100)
    plt.show()
    print(f"Saved → {fig_path.relative_to(PROJECT_ROOT)}")

In [ ]:
SEPARATION_KWARGS = dict(alpha=1.0, beta=1e-5, gamma=0.15, max_iter=300, tol=1e-3, n_inner_iter=10)

sep_rows = []
seps = {}  # (transform, inst) -> (sx_drum, sx_instr)

for transform in TRANSFORM_CONFIGS:
    for inst in tracks:
        r_drum, r_instr, r_mix = reps[(transform, inst)]
        sx, sy = separate_spectrogram(r_mix, **SEPARATION_KWARGS)

        recon_err = float(np.linalg.norm(r_mix - sx - sy))

        err_direct = l2_error(sx, r_drum) + l2_error(sy, r_instr)
        err_swap   = l2_error(sx, r_instr) + l2_error(sy, r_drum)

        if err_direct <= err_swap:
            sx_drum, sx_instr, assignment = sx, sy, "direct"
            best_err = err_direct
        else:
            sx_drum, sx_instr, assignment = sy, sx, "swapped"
            best_err = err_swap

        seps[(transform, inst)] = (sx_drum, sx_instr)

        sep_rows.append({
            "transform":            transform,
            "instrument":           inst,
            "reconstruction_error": recon_err,
            "err_direct":           err_direct,
            "err_swap":             err_swap,
            "best_error":           best_err,
            "assignment":           assignment,
        })
        print(f"[{transform}] drum + {inst:8s}  recon={recon_err:.2f}  best={best_err:.2f}  [{assignment}]")

sep_df = pd.DataFrame(sep_rows)

In [ ]:
for transform in TRANSFORM_CONFIGS:
    fig, axes = plt.subplots(4, 5, figsize=(24, 14))
    fig.suptitle(f"Drum separation — {transform.upper()}", fontsize=13)

    for row_idx, inst in enumerate(tracks):
        r_drum, r_instr, r_mix = reps[(transform, inst)]
        sx_drum, sx_instr = seps[(transform, inst)]

        for col_idx, (rep, label) in enumerate([
            (r_drum,   "R(drum)"),
            (r_instr,  f"R({inst})"),
            (r_mix,    "R(mix)"),
            (sx_drum,  "Sx → drum"),
            (sx_instr, f"Sy → {inst}"),
        ]):
            ax = axes[row_idx, col_idx]
            im = ax.imshow(
                amplitude_to_db(np.clip(rep, 0, None)),
                origin="lower", aspect="auto", interpolation="none",
            )
            plt.colorbar(im, ax=ax, format="%+2.0f dB")
            if row_idx == 0:
                ax.set_title(label, fontsize=9)
            if col_idx == 0:
                ax.set_ylabel(inst)

    plt.tight_layout()
    fig_path = RESULTS_DIR / "figures" / f"{transform}_separation.png"
    plt.savefig(fig_path, dpi=100)
    plt.show()
    print(f"Saved → {fig_path.relative_to(PROJECT_ROOT)}")

In [ ]:
csv_path = RESULTS_DIR / "metrics.csv"
df.to_csv(csv_path, index=False)
print(f"Saved → {csv_path.relative_to(PROJECT_ROOT)}")
display(df.set_index(["transform", "instrument"]))

sep_csv_path = RESULTS_DIR / "separation_metrics.csv"
sep_df.to_csv(sep_csv_path, index=False)
print(f"Saved → {sep_csv_path.relative_to(PROJECT_ROOT)}")
sep_df.set_index(["transform", "instrument"])